# Spatial Network Routing Optimization (Distributed Scenario Generation)

This notebook executes a distributed capacity-routing simulation using PySpark to evaluate large-scale network topologies, generating an optimal activation policy dataset.


In [ ]:
# Setup & Imports
import numpy as np
import random
import hashlib
import time
import os
import json
from datetime import datetime, timezone

try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("NetworkOptimization").getOrCreate()
    print("PySpark session initialized. DataFrame API available.")
    try:
        sc = spark.sparkContext
    except Exception as e:
        print("SparkContext restrito (Unity Catalog Shared Cluster). O modo DataFrame/Pandas UDF será utilizado.")
        sc = None
except Exception as e:
    print("PySpark not found. Running in local mode.")
    spark = None
    sc = None

In [ ]:
# Distributed Core Logic - Network State & Scenario Evaluator
# This code is sent to the Spark workers.

class ResourceGrid:
    def __init__(self, rows: int, cols: int):
        self.rows = rows
        self.cols = cols
        h = 2 * rows + 1
        w = 2 * cols + 1
        self.matrix = np.zeros((h, w), dtype=np.int8)
        for r in range(0, h, 2):
            for c in range(0, w, 2):
                self.matrix[r, c] = 8 # Node marker
                
    def available_links(self):
        links = []
        for r in range(self.matrix.shape[0]):
            for c in range(self.matrix.shape[1]):
                if r % 2 == 0 and c % 2 == 1:
                    if self.matrix[r, c] == 0:
                        links.append(f"H_{r}_{c}")
                elif r % 2 == 1 and c % 2 == 0:
                    if self.matrix[r, c] == 0:
                        links.append(f"V_{r}_{c}")
        return links
        
    def activate_link(self, label: str, agent: int = 1) -> int:
        _, r_str, c_str = label.split("_")
        r, c = int(r_str), int(c_str)
        if self.matrix[r, c] != 0:
            raise ValueError("Link already active")
        self.matrix[r, c] = agent
        return self._check_zones(r, c, agent)
        
    def _check_zones(self, r: int, c: int, agent: int) -> int:
        closed = 0
        for z_r, z_c in self._adjacent_zones(r, c):
            if self._zone_closed(z_r, z_c):
                self.matrix[z_r, z_c] = agent
                closed += 1
        return closed
        
    def _adjacent_zones(self, r, c):
        adj = []
        if r % 2 == 1:
            if c - 1 >= 0: adj.append((r, c - 1))
            if c + 1 < self.matrix.shape[1]: adj.append((r, c + 1))
        else:
            if r - 1 >= 0: adj.append((r - 1, c))
            if r + 1 < self.matrix.shape[0]: adj.append((r + 1, c))
        return [(br, bc) for (br, bc) in adj if br % 2 == 1 and bc % 2 == 1]
        
    def _zone_closed(self, z_r, z_c):
        up = self.matrix[z_r - 1, z_c] != 0
        down = self.matrix[z_r + 1, z_c] != 0
        left = self.matrix[z_r, z_c - 1] != 0
        right = self.matrix[z_r, z_c + 1] != 0
        return up and down and left and right
        
    def deactivate_link(self, label: str):
        _, r_str, c_str = label.split("_")
        r, c = int(r_str), int(c_str)
        for z_r, z_c in self._adjacent_zones(r, c):
            if self._zone_closed(z_r, z_c):
                self.matrix[z_r, z_c] = 0
        self.matrix[r, c] = 0
        
    def is_terminal(self):
        return len(self.available_links()) == 0

def get_canonical_links(rows, cols):
    labels = []
    h = 2 * rows + 1
    w = 2 * cols + 1
    for r in range(h):
        for c in range(w):
            if r % 2 == 0 and c % 2 == 1:
                labels.append(f"H_{r}_{c}")
            elif r % 2 == 1 and c % 2 == 0:
                labels.append(f"V_{r}_{c}")
    return labels

def evaluate_scenario(grid, depth, alpha, beta, maximizing, zones_sys=0, zones_env=0):
    if depth == 0 or grid.is_terminal():
        return zones_sys - zones_env
        
    orig_links = grid.available_links()
    good_links = []
    normal_links = []
    current_agent = 1 if maximizing else -1
    
    for l in orig_links:
        closed = grid.activate_link(l, current_agent)
        grid.deactivate_link(l)
        if closed > 0:
            good_links.append(l)
        else:
            normal_links.append(l)
            
    links = good_links + normal_links
    
    if maximizing:
        best = -10000
        for l in links:
            closed = grid.activate_link(l, 1)
            if closed > 0:
                val = evaluate_scenario(grid, depth - 1, alpha, beta, True, zones_sys + closed, zones_env)
            else:
                val = evaluate_scenario(grid, depth - 1, alpha, beta, False, zones_sys, zones_env)
            grid.deactivate_link(l)
            best = max(best, val)
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best
    else:
        best = 10000
        for l in links:
            closed = grid.activate_link(l, -1)
            if closed > 0:
                val = evaluate_scenario(grid, depth - 1, alpha, beta, False, zones_sys, zones_env + closed)
            else:
                val = evaluate_scenario(grid, depth - 1, alpha, beta, True, zones_sys, zones_env)
            grid.deactivate_link(l)
            best = min(best, val)
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best

def get_link_scores(grid, depth):
    links = grid.available_links()
    scores = {}
    for l in links:
        closed = grid.activate_link(l, 1)
        if closed > 0:
            val = evaluate_scenario(grid, depth - 1, -10001, 10001, True, closed, 0)
        else:
            val = evaluate_scenario(grid, depth - 1, -10001, 10001, False, 0, 0)
        grid.deactivate_link(l)
        scores[l] = val
    return scores

def get_optimal_routing_scores(grid, depth=7):
    if not grid.available_links():
        raise ValueError("No links available")
    scores = get_link_scores(grid, depth)
    best_val = max(scores.values())
    best_links = [l for l, v in scores.items() if v == best_val]
    return random.choice(best_links), scores


In [ ]:
# Orchestration & Progress Tracking

NUM_SAMPLES = 300000
TAMANHO_LOTE = 5000
DEPTH = 7
ROWS, COLS = 4, 3
SCORE_INDISPONIVEL = -1e9

DIRETORIO_SAIDA = "/Workspace/Users/c092820@corp.caixa.gov.br/CNN"

os.makedirs(DIRETORIO_SAIDA, exist_ok=True)

def generate_random_grid(rows, cols):
    grid = ResourceGrid(rows, cols)
    links = grid.available_links()
    total = len(links)
    min_f = int(total * 0.15)
    max_f = int(total * 0.85)
    qty = random.randint(min_f, max_f)
    random.shuffle(links)
    for l in links[:qty]:
        if l in grid.available_links():
            agent = 1 if len(grid.available_links()) % 2 == 0 else -1
            grid.activate_link(l, agent)
    return grid

def process_batch_pandas(iterator):
    # Worker function for Unity Catalog Shared Clusters (Pandas UDF)
    import pandas as pd
    import json
    
    for pdf in iterator:
        results = []
        for _ in range(len(pdf)):
            while True:
                grid = generate_random_grid(ROWS, COLS)
                if grid.is_terminal(): continue
                try:
                    best_link, scores_dict = get_optimal_routing_scores(grid, DEPTH)
                    results.append({
                        "matriz": [int(x) for x in grid.matrix.flatten().tolist()],
                        "best_link": best_link,
                        "scores_dict": json.dumps(scores_dict)
                    })
                    break
                except Exception as e:
                    pass
        yield pd.DataFrame(results)

def log_progresso(gerados: int, total: int, inicio: float):
    decorrido = time.time() - inicio
    porcentagem = gerados / total * 100
    estimativa = (decorrido / gerados * (total - gerados)) if gerados > 0 else 0
    entrada = {
        "registros_gerados": gerados,
        "total_alvo": total,
        "porcentagem": round(porcentagem, 2),
        "tempo_decorrido_s": round(decorrido, 2),
        "estimativa_restante_s": round(estimativa, 2),
    }
    print(json.dumps(entrada, ensure_ascii=False))

def vetor_scores(scores_dict, indice_label, n_labels):
    import numpy as np
    vetor = np.full(n_labels, SCORE_INDISPONIVEL, dtype=np.float32)
    for label, valor in scores_dict.items():
        vetor[indice_label[label]] = float(valor)
    return vetor

In [ ]:
# Execution Loop (Distributed Batches)
import pandas as pd
import json
import numpy as np
import time
import os
import glob

labels_canonicos = get_canonical_links(ROWS, COLS)
indice_label = {lab: i for i, lab in enumerate(labels_canonicos)}
n_labels = len(labels_canonicos)

print(f"Iniciando geração distribuída de {NUM_SAMPLES} cenários (Depth {DEPTH})...")
print(f"Destino: {DIRETORIO_SAIDA}")

# ---- LÓGICA DE RETOMADA (CHECKPOINT) ----
# Verifica se já existem arquivos .npz na pasta para continuar de onde parou.
arquivos_existentes = sorted(glob.glob(os.path.join(DIRETORIO_SAIDA, f"dataset_pequeno_*.npz")))
if arquivos_existentes:
    ultimo_arquivo = arquivos_existentes[-1]
    # Extrai o número do lote do nome do arquivo (ex: dataset_pequeno_0012.npz -> 12)
    ultimo_lote = int(ultimo_arquivo.split('_')[-1].split('.')[0])
    total_gerado = ultimo_lote * TAMANHO_LOTE
    print(f"Retomando do checkpoint: Lote {ultimo_lote} ({total_gerado} registros já gerados).")
else:
    total_gerado = 0
    ultimo_lote = 0

inicio = time.time()

LOG_INTERVAL = 1000 
schema = "matriz array<int>, best_link string, scores_dict string"

estados_lote = []
rotulos_lote = []
scores_lote = []
indices_lote = []

while total_gerado < NUM_SAMPLES:
    restante = NUM_SAMPLES - total_gerado
    chunk_atual = min(LOG_INTERVAL, restante)
    
    if spark:
        num_partitions = chunk_atual
        df = spark.range(0, chunk_atual, 1, num_partitions)
        
        res_df = df.mapInPandas(process_batch_pandas, schema=schema)
        resultados_chunk = res_df.collect()
        
        for row in resultados_chunk:
            matriz = np.array(row.matriz, dtype=np.int8).reshape(2*ROWS+1, 2*COLS+1)
            best_link = row.best_link
            scores_dict = json.loads(row.scores_dict)
            
            estados_lote.append(matriz)
            rotulos_lote.append(best_link)
            scores_lote.append(vetor_scores(scores_dict, indice_label, n_labels))
            indices_lote.append(total_gerado)
            total_gerado += 1
            
    else:
        pass
        
    log_progresso(total_gerado, NUM_SAMPLES, inicio)
        
    if len(estados_lote) >= TAMANHO_LOTE or total_gerado >= NUM_SAMPLES:
        if len(estados_lote) > 0:
            ultimo_lote += 1
            caminho_lote = os.path.join(DIRETORIO_SAIDA, f"dataset_pequeno_{ultimo_lote:04d}.npz")
            np.savez_compressed(
                caminho_lote,
                estados=np.array(estados_lote, dtype=np.int8),
                rotulos=np.array(rotulos_lote, dtype=str),
                scores=np.array(scores_lote, dtype=np.float32),
                indices=np.array(indices_lote, dtype=np.int32),
                labels_canonicos=np.array(labels_canonicos, dtype=str),
            )
            print(f"Lote {ultimo_lote:04d} salvo em {caminho_lote} — {total_gerado}/{NUM_SAMPLES} registros.")
            
            estados_lote = []
            rotulos_lote = []
            scores_lote = []
            indices_lote = []

print(f"Geração concluída: {total_gerado} registros salvos.")